# التصنيف الهجين باستخدام SVM (Hybrid Classification) — D-MorphNet

هذا الدفتر ينفّذ خطوة **"Hybrid Classification Using Support Vector Machine"**: بعد استخراج السمات العميقة بـ EfficientNet-B6، تُمرَّر إلى **SVM** ليصدر القرار النهائي — "حقيقية" أم "مدموجة" — مع **فصل تام** بين مرحلة الاستخراج ومرحلة التصنيف، وهو ما يمنح تعميماً أفضل على الصور الجديدة ويمنع الحفظ الزائد مقارنة بالنماذج العميقة من الطرف إلى الطرف.

## مسار النظام (الشكل ١ في الورقة)

```
جمع البيانات (صورة وجه RGB بحجم H×W×3)
        │
المعالجة المسبقة (تغيير الحجم إلى 528×528 لـ EfficientNet-B6)
        │
العمود الفقري EfficientNet-B6 (Compound Scaling + Batch Normalization)
        │
طبقة التجميع المتوسط الشامل (GAP)
        │
متجه السمات العميق (1D)
   ├─ الخيار A: طبقة كاملة الاتصال + Softmax
   └─ الخيار B: آلة المتجهات الداعمة (SVM)  ← ما نعتمده
        │
مقاييس التقييم (الدقة، F1، مصفوفة الالتباس)
        │
طبقة الإخراج: الفئة 0 = حقيقية ، الفئة 1 = مدموجة
```

## الأساس الرياضي لـ SVM

لكل صورة، متجه السمات المستخرج $F_i \in \mathbb{R}^{d}$ مع ملصق الفئة:

$$y_i \in \{-1, +1\}, \qquad y_i = -1 \text{ (وجه حقيقي)}, \quad y_i = +1 \text{ (وجه مدموج)}$$

يبحث SVM عن **حدّ فاصل (Hyperplane)** يفصل الفئتين في فضاء السمات:

$$w^{T}F + b = 0$$

حيث $w$ متجه الأوزان و $b$ الانحياز، ويتعلمهما بحل مسألة التحسين (تعظيم الهامش مع تقليل أخطاء التصنيف):

$$\min_{w,\,b,\,\xi}\ \frac{1}{2}\lVert w \rVert^{2} + C\sum_{i=1}^{N}\xi_i$$

$$\text{بقيد:}\quad y_i\,(w^{T}F_i + b) \ge 1 - \xi_i, \qquad \xi_i \ge 0$$

- $\xi_i$: **متغيرات التراخي (Slack)** تسمح بقدر محدود من الأخطاء.
- $C$: معامل **التنظيم** الذي يوازن بين تعظيم الهامش وتقليل الأخطاء (سنضبطه تجريبياً).
- $N$: عدد عيّنات التدريب.

وفي مرحلة الاختبار، القرار النهائي:

$$\hat{y} = \operatorname{sign}(w^{T}F + b)$$

ولأن SVM يعمل على سمات B6 عالية المستوى بدل البكسلات الخام، يكون الفصل بين الفئتين أوضح وأكثر فاعلية.


## ١) تجهيز البيئة


In [ ]:
!pip -q install scikit-learn joblib

import os, glob, random, shutil
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("✅ البيئة جاهزة")


## ٢) تحميل متجهات السمات $F_i$ من الخطوة السابقة

نحمّل السمات المحفوظة (`.npz`) من دفتر التمثيل العميق:
- الأولوية لسمات **ما بعد الضبط الدقيق** (`effb6_ft_*`)، وإن لم توجد نستخدم سمات الطبقات المجمّدة (`effb6_*`).
- نبحث محلياً أولاً ثم في مجلد النماذج على Google Drive.


In [ ]:
FEAT_DIR = "/content/DMorphNet_features"
DRIVE_DIR = "/content/drive/MyDrive/DMorphNet_models"
SPLITS = ["train", "val", "test"]


def find_npz(name):
    for root in [FEAT_DIR, DRIVE_DIR]:
        p = os.path.join(root, name)
        if os.path.exists(p):
            return p
    return None


if find_npz("effb6_ft_train.npz") is None and find_npz("effb6_train.npz") is None:
    from google.colab import drive
    drive.mount('/content/drive')

PREFIX = "effb6_ft" if find_npz("effb6_ft_train.npz") else "effb6"
print("السمات المستخدمة:",
      "ما بعد الضبط الدقيق" if PREFIX == "effb6_ft" else "الطبقات المجمّدة")

F, y = {}, {}
for split in SPLITS:
    path = find_npz(f"{PREFIX}_{split}.npz")
    assert path, f"لم يتم العثور على سمات {split} — شغّل دفتر التمثيل العميق أولاً"
    data = np.load(path)
    F[split] = data["X"].astype(np.float32)
    # تحويل الملصقات إلى صيغة الورقة: -1 حقيقية ، +1 مدموجة
    y[split] = np.where(data["y"] == 0, -1, +1)
    n_real = int((y[split] == -1).sum())
    n_morph = int((y[split] == +1).sum())
    print(f"{split}: F بشكل {F[split].shape} — حقيقية(-1)={n_real} مدموجة(+1)={n_morph}")

d = F["train"].shape[1]
print(f"\nبُعد فضاء السمات d = {d}")


## ٣) توحيد المقاييس وضبط معامل التنظيم $C$

- نوحّد مقاييس السمات بـ `StandardScaler` (متوسط 0 وانحراف 1) — ضروري حتى لا تهيمن سمات ذات مدى كبير على حساب الهامش.
- نجرّب عدة قيم لـ $C$ وندرّب SVM **خطياً** (مطابقاً للصيغة $w^TF+b$) على التدريب، ونختار القيمة الأفضل على مجموعة **التحقق**:
  - $C$ صغير ← هامش أوسع وتسامح أكبر مع الأخطاء ($\xi_i$ أكبر).
  - $C$ كبير ← أخطاء أقل على التدريب لكن خطر حفظ زائد.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

scaler = StandardScaler().fit(F["train"])
Fs = {s: scaler.transform(F[s]) for s in SPLITS}

C_GRID = [0.01, 0.1, 1.0, 10.0]
results_C = {}
for C in C_GRID:
    clf = LinearSVC(C=C, random_state=SEED)
    clf.fit(Fs["train"], y["train"])
    acc = accuracy_score(y["val"], clf.predict(Fs["val"]))
    results_C[C] = acc
    print(f"C = {C:<6} → دقة التحقق = {acc:.4f}")

BEST_C = max(results_C, key=results_C.get)
print(f"\n🏆 أفضل معامل تنظيم: C = {BEST_C}")


## ٤) تدريب SVM النهائي

ندرّب المصنّف النهائي بأفضل $C$ على كامل سمات التدريب (30,000 متجه). حل مسألة التحسين يعطينا $w \in \mathbb{R}^{2304}$ و $b$ — وهما كل ما يحتاجه القرار النهائي.


In [ ]:
svm_final = LinearSVC(C=BEST_C, random_state=SEED)
svm_final.fit(Fs["train"], y["train"])

w = svm_final.coef_[0]        # متجه الأوزان w
b = svm_final.intercept_[0]   # الانحياز b
print(f"شكل متجه الأوزان w: {w.shape}")
print(f"الانحياز b = {b:.6f}")
print(f"معيار الهامش ||w|| = {np.linalg.norm(w):.4f}")
print("✅ اكتمل تدريب SVM النهائي")


## ٥) دالة القرار: $\hat{y} = \operatorname{sign}(w^{T}F + b)$

نتحقق عملياً أن التنبؤ هو فعلاً إشارة دالة القرار — نحسب $w^{T}F + b$ **يدوياً** بضرب المصفوفات ونقارن مع `predict` الخاص بالمكتبة: يجب أن يتطابقا 100%.


In [ ]:
# الحساب اليدوي لدالة القرار على مجموعة الاختبار
decision_manual = Fs["test"] @ w + b          # w^T F + b لكل صورة
y_hat_manual = np.sign(decision_manual)       # sign(...)
y_hat_sklearn = svm_final.predict(Fs["test"])

match = (y_hat_manual == y_hat_sklearn).mean()
print(f"تطابق الحساب اليدوي مع المكتبة: {match:.2%}")
assert match == 1.0

# توزيع قيم دالة القرار للفئتين — كلما ابتعدت القيم عن الصفر كان الفصل أوضح
plt.figure(figsize=(9, 4))
plt.hist(decision_manual[y["test"] == -1], bins=60, alpha=0.6,
         label="حقيقية (y = -1)")
plt.hist(decision_manual[y["test"] == +1], bins=60, alpha=0.6,
         label="مدموجة (y = +1)")
plt.axvline(0, color="black", linestyle="--", label="الحد الفاصل wᵀF+b=0")
plt.xlabel("قيمة دالة القرار wᵀF + b")
plt.ylabel("عدد الصور")
plt.title("فصل الفئتين في فضاء السمات (مجموعة الاختبار)")
plt.legend()
plt.tight_layout()
plt.show()


## ٦) لماذا الخيار B؟ مقارنة SVM مع طبقة Softmax (الخيار A)

الشكل ١ يعرض خيارين لتصنيف متجه السمات:
- **الخيار A**: طبقة كاملة الاتصال + Softmax (امتداد للتعلم العميق).
- **الخيار B**: SVM (ما تعتمده الورقة).

ندرّب الخيار A على **نفس السمات تماماً** ونقارن على مجموعة الاختبار — الفجوة بين دقة التدريب والاختبار تكشف أيهما يعمّم أفضل ولماذا الفصل بين الاستخراج والتصنيف يمنع الحفظ الزائد.


In [ ]:
import tensorflow as tf

# الخيار A: رأس كامل الاتصال + Softmax على نفس السمات
y01 = {s: (y[s] == 1).astype(np.int32) for s in SPLITS}   # صيغة 0/1 لـ Softmax

head = tf.keras.Sequential([
    tf.keras.layers.Input((Fs["train"].shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2, activation="softmax"),
])
head.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
             loss="sparse_categorical_crossentropy", metrics=["accuracy"])
head.fit(Fs["train"], y01["train"], epochs=10, batch_size=256,
         validation_data=(Fs["val"], y01["val"]), verbose=2)

acc_A_train = head.evaluate(Fs["train"], y01["train"], verbose=0)[1]
acc_A_test  = head.evaluate(Fs["test"],  y01["test"],  verbose=0)[1]

acc_B_train = accuracy_score(y["train"], svm_final.predict(Fs["train"]))
acc_B_test  = accuracy_score(y["test"],  svm_final.predict(Fs["test"]))

print("\n═══ المقارنة على نفس السمات ═══")
print(f"الخيار A (FC + Softmax): تدريب={acc_A_train:.4f}  اختبار={acc_A_test:.4f}  "
      f"الفجوة={(acc_A_train - acc_A_test)*100:.2f} نقطة")
print(f"الخيار B (SVM):          تدريب={acc_B_train:.4f}  اختبار={acc_B_test:.4f}  "
      f"الفجوة={(acc_B_train - acc_B_test)*100:.2f} نقطة")
print("\nالفجوة الأصغر بين التدريب والاختبار = تعميم أفضل وحفظ زائد أقل")


## ٧) التقييم النهائي: الدقة و F1 ومصفوفة الالتباس

نفس مقاييس الشكل ١: **Accuracy** و **F1-Score** و **Confusion Matrix**، على مجموعتي التحقق والاختبار (هويات لم يرها النظام إطلاقاً).


In [ ]:
from sklearn.metrics import (f1_score, classification_report,
                             confusion_matrix)

for split in ["val", "test"]:
    pred = svm_final.predict(Fs[split])
    acc = accuracy_score(y[split], pred)
    f1 = f1_score(y[split], pred, pos_label=+1)
    print(f"\n===== {split} =====")
    print(f"الدقة (Accuracy) = {acc:.4f}   مقياس F1 = {f1:.4f}")
    print(classification_report(y[split], pred,
                                target_names=["حقيقية (-1)", "مدموجة (+1)"]))

# مصفوفة الالتباس لمجموعة الاختبار
pred_test = svm_final.predict(Fs["test"])
cm = confusion_matrix(y["test"], pred_test, labels=[-1, +1])

plt.figure(figsize=(5, 4.5))
plt.imshow(cm, cmap="Blues")
plt.title("مصفوفة الالتباس — الاختبار")
plt.xticks([0, 1], ["حقيقية", "مدموجة"])
plt.yticks([0, 1], ["حقيقية", "مدموجة"])
plt.xlabel("التوقع")
plt.ylabel("الحقيقة")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()


## ٨) دالة التنبؤ الكاملة لصورة واحدة (النظام النهائي)

نجمّع المسار كاملاً في دالة واحدة:

**صورة** ← معالجة (528×528) ← $f_{B6}$ ← GAP ← $F$ ← توحيد المقاييس ← $\operatorname{sign}(w^TF+b)$ ← **الفئة 0: حقيقية / الفئة 1: مدموجة**

ونجرّبها على عيّنات من مجموعة الاختبار (تحتاج نموذج السمات المحفوظ من دفتر التمثيل العميق، أو تبني B6 مجمّداً كبديل).


In [ ]:
import cv2
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = 528
FEAT_MODEL_PATH = os.path.join(DRIVE_DIR, "effb6_finetuned_features.keras")

if os.path.exists(FEAT_MODEL_PATH):
    feat_model = tf.keras.models.load_model(FEAT_MODEL_PATH)
    print("تم تحميل نموذج السمات المضبوط دقيقاً")
else:
    # بديل: B6 مجمّد بأوزان ImageNet
    from tensorflow.keras.applications import EfficientNetB6
    feat_model = EfficientNetB6(include_top=False, weights="imagenet",
                                pooling="avg",
                                input_shape=(IMG_SIZE, IMG_SIZE, 3))
    print("تم بناء B6 مجمّد (لم يُعثر على النموذج المضبوط)")

CLASS_NAMES = {-1: "الفئة 0: حقيقية (Real)", +1: "الفئة 1: مدموجة (Morph)"}


def predict_image(img_path):
    # المسار الكامل: صورة ← B6 ← GAP ← F ← Scaler ← sign(wF+b)
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    feat = feat_model.predict(preprocess_input(rgb)[None, ...], verbose=0)
    feat_s = scaler.transform(feat)
    decision = float(feat_s @ w + b)
    y_hat = int(np.sign(decision)) if decision != 0 else 1
    return y_hat, decision


# تجربة على عيّنات من مجموعة الاختبار المعالجة
PROC_DIR = "/content/DMorphNet_processed"
samples = (random.sample(glob.glob(os.path.join(PROC_DIR, "test", "real", "*.jpg")), 2)
           + random.sample(glob.glob(os.path.join(PROC_DIR, "test", "morph", "*.jpg")), 2))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, samples):
    y_hat, dec = predict_image(p)
    truth = "حقيقية" if "/real/" in p else "مدموجة"
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    ax.set_title(f"الحقيقة: {truth}\nالتنبؤ: {CLASS_NAMES[y_hat].split(':')[1]}"
                 f"\nwᵀF+b = {dec:+.2f}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()


## ٩) حفظ المصنّف النهائي في Google Drive

نحفظ SVM النهائي (بأفضل $C$) والـ Scaler — مع النموذج المحفوظ سابقاً يصبح نظام D-MorphNet كاملاً وقابلاً للنشر.


In [ ]:
import joblib
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DRIVE_DIR, exist_ok=True)
joblib.dump(svm_final, os.path.join(DRIVE_DIR, "svm_hybrid_final.joblib"))
joblib.dump(scaler, os.path.join(DRIVE_DIR, "scaler_hybrid_final.joblib"))
np.savez(os.path.join(DRIVE_DIR, "svm_hyperplane.npz"), w=w, b=b, C=BEST_C)

print("تم الحفظ في:", DRIVE_DIR)
print(os.listdir(DRIVE_DIR))
print("\n🎉 اكتمل التصنيف الهجين: EfficientNet-B6 (سمات) + SVM (قرار نهائي)")
